# Component I: Vector Indexing and Embedding

This part prepares the uploaded PDF for retrieval. It extracts the text, splits it into chunks, creates embeddings, and stores them in FAISS. It also exports the chunks to Excel so we can check the results.

### Step 1: Install required libraries

This cell installs the packages needed for PDF reading, chunking, embeddings, FAISS retrieval, the Excel export, and the SEA-LION language model.

In [1]:
# Install the packages used for PDF processing and RAG.

!pip install -q \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    transformers \
    sentencepiece \
    pandas \
    openpyxl \
    bitsandbytes \
    accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.0 MB/s eta 0:00:00


### Step 2: Import libraries

This cell imports the tools used throughout the notebook. These include PDF extraction, text cleaning, embeddings, FAISS, data handling, and Excel formatting.

In [2]:
# Import the libraries used in the indexing steps.

from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

import faiss
import numpy as np
import pandas as pd
import pickle
import re
import os

from openpyxl.styles import Font, PatternFill, Alignment

### Step 3: Set the indexing parameters

This cell sets the chunk size, overlap, number of retrieved chunks, and embedding model. We can change `CHUNK_SIZE` and `CHUNK_OVERLAP` for the experiment while keeping `TOP_K` and the embedding model the same.

In [3]:
# Set the chunking and retrieval settings.

# Set the chunk size and overlap for each experiment.

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
TOP_K = 3

# Use the same embedding model for all runs.
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

### Step 4: Validate the settings

This cell checks that chunk size and `TOP_K` are greater than zero and that overlap is not negative or equal to or larger than the chunk size. It stops the program if the values are invalid and prints the settings being used.

In [4]:
# Check the indexing settings before running.

if CHUNK_SIZE <= 0:
    raise ValueError("CHUNK_SIZE must be greater than 0.")

if CHUNK_OVERLAP < 0:
    raise ValueError("CHUNK_OVERLAP cannot be negative.")

if CHUNK_OVERLAP >= CHUNK_SIZE:
    raise ValueError(
        "CHUNK_OVERLAP must be smaller than CHUNK_SIZE."
    )

if TOP_K <= 0:
    raise ValueError("TOP_K must be greater than 0.")


print("=" * 70)
print("RAG INDEXING CONFIGURATION")
print("=" * 70)

print("Chunk size       :", CHUNK_SIZE)
print("Chunk overlap    :", CHUNK_OVERLAP)
print("Top-K             :", TOP_K)
print("Embedding model   :", EMBEDDING_MODEL_NAME)

RAG INDEXING CONFIGURATION
Chunk size       : 1000
Chunk overlap    : 100
Top-K             : 3
Embedding model   : all-MiniLM-L6-v2


### Step 5: Upload the PDF

This cell uploads the document to Colab and checks whether a PDF was selected. The notebook uses the first uploaded file, so we should upload only our assigned member PDF.

In [5]:
# Upload and select a PDF file.

print("\n")
print("=" * 70)
print("UPLOAD DOCUMENT")
print("=" * 70)

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("No file was uploaded.")

# Use the first uploaded PDF.
pdf_file = next(iter(uploaded.keys()))

if not pdf_file.lower().endswith(".pdf"):
    raise ValueError(
        "The uploaded file is not a PDF. Please upload a PDF."
    )

print("\nSelected file:")
print(pdf_file)



UPLOAD DOCUMENT


Saving LendSuri_Member_2_Corpus.pdf to LendSuri_Member_2_Corpus.pdf

Selected file:
LendSuri_Member_2_Corpus.pdf


### Step 6: Extract text from the PDF

This cell reads the PDF page by page using `PdfReader`. It extracts the text, removes extra whitespace, and saves the page number with the text. It also counts how many pages contain extractable text.

In [6]:
# Extract and clean the text from each PDF page.

print("\n")
print("=" * 70)
print("EXTRACTING PDF TEXT")
print("=" * 70)

reader = PdfReader(pdf_file)

pages = []

for page_number, page in enumerate(reader.pages, start=1):

    page_text = page.extract_text()

    if page_text:

        # Remove extra spaces and line breaks.
        page_text = re.sub(
            r"\s+",
            " ",
            page_text
        ).strip()

        if page_text:
            pages.append({
                "page": page_number,
                "text": page_text
            })


print("PDF pages:", len(reader.pages))
print("Pages containing text:", len(pages))



EXTRACTING PDF TEXT
PDF pages: 126
Pages containing text: 126


### Step 7: Check the extracted text

This cell stops the process if no text was extracted from the PDF. This can happen when the document contains scanned images instead of selectable text.

In [7]:
# Stop if the PDF has no extractable text.

if len(pages) == 0:
    raise RuntimeError(
        "No text could be extracted from the PDF. "
        "The PDF may consist of scanned images and require OCR."
    )

### Step 8: Create text chunks

The `create_chunks` function divides each page into character-based chunks using the selected size and overlap. Each chunk keeps its own ID and original page number. The next chunk starts after the chunk size minus the overlap, and chunking starts again on each new page.

In [8]:
# Split each page into chunks and keep the page numbers.

def create_chunks(
    pages,
    chunk_size,
    chunk_overlap
):
    """
    Split each page into character-based chunks.

    Chunking is performed independently for each page.
    Overlap is maintained between consecutive chunks
    from the same page.
    """

    chunks = []
    chunk_id = 0

    step_size = chunk_size - chunk_overlap

    for page_data in pages:

        page_number = page_data["page"]
        page_text = page_data["text"]

        start = 0

        while start < len(page_text):

            end = start + chunk_size

            chunk_text = page_text[start:end].strip()

            if chunk_text:
                chunks.append({
                    "chunk_id": chunk_id,
                    "page": page_number,
                    "text": chunk_text
                })

                chunk_id += 1

            # Move forward and keep the selected overlap.
            start += step_size

    return chunks


chunks = create_chunks(
    pages,
    CHUNK_SIZE,
    CHUNK_OVERLAP
)

### Step 9: Check the chunking results

This cell shows the total number of chunks and their average, shortest, and longest character lengths. It also checks that chunking produced at least one result.

In [9]:
# Show the number and length of the generated chunks.

print("\n")
print("=" * 70)
print("CHUNKING RESULTS")
print("=" * 70)

print("Number of chunks:", len(chunks))

if len(chunks) == 0:
    raise RuntimeError(
        "No chunks were generated. Check the extracted PDF text."
    )

chunk_lengths = [
    len(chunk["text"])
    for chunk in chunks
]

print(
    "Average characters per chunk:",
    round(np.mean(chunk_lengths), 2)
)

print(
    "Shortest chunk:",
    min(chunk_lengths),
    "characters"
)

print(
    "Longest chunk:",
    max(chunk_lengths),
    "characters"
)



CHUNKING RESULTS
Number of chunks: 252
Average characters per chunk: 702.81
Shortest chunk: 137 characters
Longest chunk: 1000 characters


### Step 10: Export chunks to Excel

This cell makes an Excel file containing every chunk's ID, source PDF, page number, character count, settings, and text. It formats the worksheet and downloads it so we can inspect whether the chunks contain useful information.

In [10]:
# Save the generated chunks in an Excel file.

print("\n")
print("=" * 70)
print("EXPORTING GENERATED CHUNKS TO EXCEL")
print("=" * 70)

# Put each chunk and its details in one Excel row.
chunk_records = []

for chunk in chunks:

    chunk_records.append({
        "Chunk_ID": chunk["chunk_id"],
        "Source_Document": pdf_file,
        "Page_Number": chunk["page"],
        "Character_Count": len(chunk["text"]),
        "Chunk_Size_Setting": CHUNK_SIZE,
        "Chunk_Overlap_Setting": CHUNK_OVERLAP,
        "Chunk_Text": chunk["text"]
    })

chunks_df = pd.DataFrame(chunk_records)

# Use the PDF name for the Excel filename.
base_name = os.path.splitext(
    os.path.basename(pdf_file)
)[0]

EXCEL_FILE = f"{base_name}_chunks_inspection.xlsx"

# Create the Excel file and format its worksheet.
with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    chunks_df.to_excel(
        writer,
        sheet_name="Generated_Chunks",
        index=False
    )

    worksheet = writer.sheets["Generated_Chunks"]

    # Format the column headings.
    for cell in worksheet[1]:

        cell.font = Font(
            bold=True,
            color="FFFFFF"
        )

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    # Adjust the column widths.
    column_widths = {
        "A": 12,
        "B": 35,
        "C": 14,
        "D": 18,
        "E": 20,
        "F": 23,
        "G": 80
    }

    for column, width in column_widths.items():
        worksheet.column_dimensions[column].width = width

    # Make the rows easier to read.
    for row in worksheet.iter_rows(
        min_row=2,
        max_row=worksheet.max_row
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=(cell.column == 7)
            )

    # Keep the column headings visible when scrolling.
    worksheet.freeze_panes = "A2"

    # Allow filtering of Excel rows.
    worksheet.auto_filter.ref = worksheet.dimensions

    # Set the heading row height.
    worksheet.row_dimensions[1].height = 32

print("Excel file created:", EXCEL_FILE)
print("Chunks exported:", len(chunks_df))
print("Excel worksheet: Generated_Chunks")

print("\nExcel export preview:")

print(
    chunks_df[
        [
            "Chunk_ID",
            "Page_Number",
            "Character_Count"
        ]
    ].head(10).to_string(index=False)
)

# Download the Excel file from Colab.
files.download(EXCEL_FILE)



EXPORTING GENERATED CHUNKS TO EXCEL
Excel file created: LendSuri_Member_2_Corpus_chunks_inspection.xlsx
Chunks exported: 252
Excel worksheet: Generated_Chunks

Excel export preview:
 Chunk_ID  Page_Number  Character_Count
        0            1             1000
        1            1              626
        2            2             1000
        3            2              496
        4            3             1000
        5            3              629
        6            4             1000
        7            4              538
        8            5             1000
        9            5              496


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Step 11: Display sample chunks

This cell prints the first three chunks with their IDs, page numbers, lengths, and text. We can use this preview to check the chunking output before creating embeddings.

In [11]:
# Preview the first three chunks.

print("\n")
print("=" * 70)
print("SAMPLE CHUNKS")
print("=" * 70)

for chunk in chunks[:3]:

    print(f"\nChunk ID : {chunk['chunk_id']}")
    print(f"Page     : {chunk['page']}")
    print(f"Characters: {len(chunk['text'])}")

    print("-" * 70)

    print(chunk["text"])



SAMPLE CHUNKS

Chunk ID : 0
Page     : 1
Characters: 1000
----------------------------------------------------------------------
LendSuri | Researcher-prepared record from the existing v5 corpus 1 LendSuri / Member 2 A Current OLP registry snapshot (2026-01-26) Record ID: CUR2026_0005 Record date: 2026-01-26 Platform or entity: https://www.asteria.com.ph/ Company: Asteria Lending Inc. Doing Business under the name and style of Cashbazaar Evidence category: CURRENT_RECORDED_OLP_SNAPSHOT_2026 Classification: RECORDED_IN_2026_01_26_SNAPSHOT Evidence priority: CURRENT_POSITIVE_SNAPSHOT RAG source text (v5 normalized text) SEC OLP current-registry snapshot dated 2026-01-26. Platform: https://www.asteria.com.ph/. Company: Asteria Lending Inc. Doing Business under the name and style of Cashbazaar. SEC registration no.: CS201603853. Form 1 submission date(s): 07 October, 2019. Remarks: none stated. IMPORTANT: use this as dated 2026 positive registry evidence. Later SEC enforcement/advisory e

### Step 12: Load the embedding model

This cell loads `all-MiniLM-L6-v2` through Sentence Transformers. The model will convert the text chunks and user questions into vectors for similarity search.

In [12]:
# Load the embedding model.

print("\n")
print("=" * 70)
print("LOADING EMBEDDING MODEL")
print("=" * 70)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding model:", EMBEDDING_MODEL_NAME)



LOADING EMBEDDING MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: all-MiniLM-L6-v2


### Step 13: Get the chunk text

This cell collects only the text from each chunk into a list. The text list is used as the input for the embedding model.

In [13]:
# Get the text from each chunk.

chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

### Step 14: Generate embeddings

This cell converts each chunk into a numerical vector using the embedding model. It normalizes the vectors and changes them to `float32`, which FAISS uses. The matrix size is printed for checking.

In [14]:
# Generate normalized embeddings for the chunks.

print("\n")
print("=" * 70)
print("CREATING DOCUMENT EMBEDDINGS")
print("=" * 70)

embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Convert vectors to float32 for FAISS.
embeddings = embeddings.astype("float32")

print("\nEmbedding matrix shape:")
print(embeddings.shape)



CREATING DOCUMENT EMBEDDINGS


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Embedding matrix shape:
(252, 384)


### Step 15: Create the FAISS vector store

This cell builds a FAISS index using the embedding dimensions and adds the chunk vectors. It uses inner product similarity on normalized vectors to find related text during retrieval.

In [15]:
# Store the chunk embeddings in FAISS.

print("\n")
print("=" * 70)
print("CREATING FAISS VECTOR STORE")
print("=" * 70)

embedding_dimension = embeddings.shape[1]

# Use inner product similarity with normalized vectors.
index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(embeddings)

print("Embedding dimension:", embedding_dimension)
print("Vectors stored:", index.ntotal)



CREATING FAISS VECTOR STORE
Embedding dimension: 384
Vectors stored: 252


### Step 16: Link vectors to chunks

This cell creates a lookup dictionary that connects each position in the FAISS index to the original chunk. This lets us retrieve the text and page number after a search.

In [16]:
# Connect FAISS positions to their original chunks.

# Match each vector position with its chunk.

chunk_lookup = {
    position: chunk
    for position, chunk in enumerate(chunks)
}

### Step 17: Define the retrieval function

The `retrieve_documents` function converts a question into an embedding and searches FAISS for the closest chunks. It returns each result's rank, chunk ID, page number, similarity score, and text.

In [17]:
# Define how questions are matched with document chunks.

def retrieve_documents(
    query,
    top_k=3
):
    """
    Retrieve the top matching chunks using FAISS.

    Returns chunk ID, page, similarity score, and text.
    """

    if not query or not query.strip():
        raise ValueError("The query cannot be empty.")

    if top_k <= 0:
        raise ValueError("top_k must be greater than 0.")

    # Convert the question into a vector.
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = query_embedding.astype("float32")

    # Limit the results to the number of stored vectors.
    search_k = min(top_k, index.ntotal)

    # Find the closest vectors in FAISS.
    scores, indices = index.search(
        query_embedding,
        search_k
    )

    results = []

    for rank, idx in enumerate(indices[0]):

        if idx == -1:
            continue

        chunk = chunk_lookup[int(idx)]

        results.append({
            "rank": rank + 1,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": float(scores[0][rank]),
            "text": chunk["text"]
        })

    return results

### Step 18: Test document retrieval

This cell lets us enter a sample question and shows the highest-ranked chunks found by FAISS. We can compare their page numbers and text with the expected supporting evidence.

In [18]:
# Run a sample retrieval question.

print("\n")
print("=" * 70)
print("TESTING VECTOR STORE")
print("=" * 70)

test_query = input(
    "\nEnter a test question "
    "(or press Enter to skip): "
)

if test_query.strip():

    test_results = retrieve_documents(
        test_query,
        TOP_K
    )

    print("\n")
    print("=" * 70)
    print("RETRIEVED CHUNKS")
    print("=" * 70)

    for result in test_results:

        print(f"\nRESULT {result['rank']}")
        print(f"Chunk ID   : {result['chunk_id']}")
        print(f"Page       : {result['page']}")
        print(f"Similarity : {result['score']:.4f}")

        print("-" * 70)

        print(result["text"])



TESTING VECTOR STORE

Enter a test question (or press Enter to skip): 


### Step 19: Save the FAISS index

This cell saves the vector index as `rag_faiss.index`. The saved index contains the embeddings for later retrieval.

In [19]:
# Save the FAISS index.

FAISS_FILE = "rag_faiss.index"

faiss.write_index(
    index,
    FAISS_FILE
)

print("\nFAISS index saved:", FAISS_FILE)


FAISS index saved: rag_faiss.index


### Step 20: Save chunks and metadata

This cell saves the chunks and their page numbers in `rag_chunks.pkl`. These details are needed to identify the text linked to each FAISS result.

In [20]:
# Save the chunks and their metadata.

CHUNKS_FILE = "rag_chunks.pkl"

with open(CHUNKS_FILE, "wb") as f:

    pickle.dump(
        chunks,
        f
    )

print("Chunks saved:", CHUNKS_FILE)

Chunks saved: rag_chunks.pkl


### Step 21: Save the experiment settings

This cell saves the chunk size, overlap, top-k value, embedding model, and PDF filename in `rag_config.pkl`. We can use this file to identify the settings of a particular run.

In [21]:
# Save the indexing settings.

CONFIG_FILE = "rag_config.pkl"

config = {
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "top_k": TOP_K,
    "embedding_model": EMBEDDING_MODEL_NAME,
    "source_document": pdf_file
}

with open(CONFIG_FILE, "wb") as f:

    pickle.dump(
        config,
        f
    )

print("Configuration saved:", CONFIG_FILE)

Configuration saved: rag_config.pkl


### Step 22: Display the indexing summary

This cell prints the document name, number of chunks and vectors, selected settings, and saved files. It confirms that the index is ready for retrieval.

In [22]:
# Print the indexing summary.

print("\n")
print("=" * 70)
print("INDEXING COMPLETE")
print("=" * 70)

print("Source document :", pdf_file)
print("Number of chunks:", len(chunks))
print("Vectors stored  :", index.ntotal)
print("Chunk size      :", CHUNK_SIZE)
print("Chunk overlap   :", CHUNK_OVERLAP)
print("Embedding model :", EMBEDDING_MODEL_NAME)

print("\nSaved files:")
print("  ", EXCEL_FILE)
print("  ", FAISS_FILE)
print("  ", CHUNKS_FILE)
print("  ", CONFIG_FILE)

print("\nThe vector store is ready for retrieval.")
print(
    "Inspect the downloaded Excel workbook to evaluate "
    "the generated chunks."
)



INDEXING COMPLETE
Source document : LendSuri_Member_2_Corpus.pdf
Number of chunks: 252
Vectors stored  : 252
Chunk size      : 1000
Chunk overlap   : 100
Embedding model : all-MiniLM-L6-v2

Saved files:
   LendSuri_Member_2_Corpus_chunks_inspection.xlsx
   rag_faiss.index
   rag_chunks.pkl
   rag_config.pkl

The vector store is ready for retrieval.
Inspect the downloaded Excel workbook to evaluate the generated chunks.


# Component II: Prompt and Retrieval

This part loads the SEA-LION language model and uses the chunks retrieved by FAISS to answer questions. It sets the response rules, builds a prompt with the available evidence, and displays the answer and retrieved sources.

### Step 1: Import language model libraries

This cell imports Transformers and PyTorch. These are used to load the SEA-LION tokenizer and model, set 4-bit quantization, and generate answers.

In [23]:
# Import the libraries needed to run SEA-LION.

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

import torch
import re

### Step 2: Check the device

This cell checks whether Colab has an available GPU. The notebook stops if CUDA is not available because the configured SEA-LION model requires a GPU for this setup.

In [24]:
# Check if a GPU is available.

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 70)
print("RAG PROMPT PROCESSING")
print("=" * 70)

print("Device:", device)

if device != "cuda":
    raise RuntimeError(
        "No GPU found. Llama-SEA-LION-v3-8B-IT needs a GPU. "
        "In Colab, go to Runtime > Change runtime type > T4 GPU, "
        "then run all cells again."
    )

print("GPU:", torch.cuda.get_device_name(0))

RAG PROMPT PROCESSING
Device: cuda
GPU: NVIDIA L4


### Step 3: Load SEA-LION

This cell sets `Llama-SEA-LION-v3-8B-IT` as the language model and loads its tokenizer. It also configures 4-bit loading to reduce GPU memory use, sets the input and output token limits, and prepares the model for inference.

In [25]:
# Load the SEA-LION model and tokenizer.

LLM_NAME = "aisingapore/Llama-SEA-LION-v3-8B-IT"

MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 256

# Keep a few tokens available within the input limit.
RESERVED_TOKEN_MARGIN = 16

# Use 4-bit loading to reduce GPU memory use.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("\nLoading language model:")
print(LLM_NAME)

tokenizer = AutoTokenizer.from_pretrained(
    LLM_NAME
)

# Use the end token as padding when no pad token is set.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

llm.eval()

print("Language model loaded.")
print(
    "GPU memory used:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)


Loading language model:
aisingapore/Llama-SEA-LION-v3-8B-IT


config.json:   0%|          | 0.00/874 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Language model loaded.
GPU memory used: 5.41 GB


### Step 4: Set response rules and token counting

This cell defines the system instructions for LendSuri. The model is told to answer only from retrieved evidence, compare dated records carefully, give page citations, and avoid unsupported claims. The helper functions format the conversation for SEA-LION and count how many tokens the prompt uses.

In [26]:
# Set the chatbot rules and token counting functions.

# Send these rules to the model for every question.
SYSTEM_MESSAGE = ("""
You are LendSuri. Answer the question using only the provided Evidence.

Identify the exact online lending platform or company asked about.
Give a short answer in 1 to 3 sentences and cite the supporting
PDF page numbers, such as (Page 8).

Do not use outside information, guess, or claim a platform is safe
or legally authorized.

Answer in the same language as the question (English or Filipino).

If the evidence is not enough, reply exactly:
I cannot find the answer in the provided document.
""")


def build_messages(user_prompt):
    """Wrap the user prompt with the system rules as a chat."""

    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_prompt}
    ]


def to_chat_text(user_prompt):
    """Apply SEA-LION's chat template to the prompt."""

    return tokenizer.apply_chat_template(
        build_messages(user_prompt),
        tokenize=False,
        add_generation_prompt=True
    )


def count_tokens(text):

    return len(
        tokenizer.encode(
            text,
            add_special_tokens=True,
            truncation=False
        )
    )


def count_chat_tokens(user_prompt):
    """Count tokens the model actually receives (rules + template)."""

    # Do not add another start token to the chat template.
    return len(
        tokenizer(
            to_chat_text(user_prompt),
            add_special_tokens=False
        )["input_ids"]
    )

### Step 5: Order retrieved chunks

The `prioritize_documents` function sorts retrieved chunks by similarity score, from highest to lowest. This puts the closest text matches first when building the prompt.

In [27]:
# Sort retrieved chunks by similarity score.

def prioritize_documents(question, retrieved_documents):
    """
    Preserve FAISS similarity ranking.

    Assumes higher scores mean greater similarity,
    as with normalized embeddings and IndexFlatIP.
    """

    return sorted(
        retrieved_documents,
        key=lambda result: result.get("score", float("-inf")),
        reverse=True
    )

### Step 6: Build the RAG prompt

The `create_rag_prompt` function combines the question with the retrieved chunks and their page and chunk IDs. It checks how much space is available within the input token limit and includes the evidence that fits. It then prints prompt details so we can see which evidence was sent to the model.

In [28]:
# Build a prompt with retrieved evidence within the token limit.

def create_rag_prompt(
    question,
    retrieved_documents
):
    """
    Build the user message: the question plus as much retrieved
    evidence as the input budget allows. The guardrails are sent
    separately as the system message (SYSTEM_MESSAGE).

    Context is trimmed only at token boundaries.
    """

    ranked_documents = prioritize_documents(
        question,
        retrieved_documents
    )

    # Put the question before the evidence.
    prompt_prefix = (
        f"Question: {question}\n\n"
        "Evidence:\n"
    )

    prompt_suffix = (
        "\n\nAnswer the question using only the evidence above."
        "\nAnswer:"
    )

    # Count the tokens needed for instructions and the prompt.
    fixed_tokens = count_chat_tokens(
        prompt_prefix + prompt_suffix
    )

    available_context_tokens = (
        MAX_INPUT_TOKENS
        - fixed_tokens
        - RESERVED_TOKEN_MARGIN
    )

    if available_context_tokens <= 0:
        raise ValueError(
            "The question and instructions leave no room "
            "for retrieved evidence. Shorten the question."
        )

    selected_parts = []
    remaining_tokens = available_context_tokens

    for result in ranked_documents:

        if remaining_tokens <= 0:
            break

        source_text = (
            f"[Page {result['page']}, "
            f"Chunk {result['chunk_id']}]\n"
            f"{result['text']}"
        )

        # Count the tokens in each retrieved chunk.
        source_tokens = tokenizer.encode(
            source_text,
            add_special_tokens=False
        )

        separator_tokens = (
            tokenizer.encode(
                "\n\n",
                add_special_tokens=False
            )
            if selected_parts
            else []
        )

        tokens_available = (
            remaining_tokens - len(separator_tokens)
        )

        if tokens_available <= 0:
            break

        # Include as much of the chunk as fits.
        included_tokens = source_tokens[
            :tokens_available
        ]

        if not included_tokens:
            continue

        included_text = tokenizer.decode(
            included_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        selected_parts.append(included_text)

        remaining_tokens -= (
            len(separator_tokens)
            + len(included_tokens)
        )

        # Stop when a chunk is cut to fit the token limit.
        if len(included_tokens) < len(source_tokens):
            break

    context = "\n\n".join(selected_parts)

    prompt = (
        prompt_prefix
        + context
        + prompt_suffix
    )

    # Check the final prompt length.
    final_token_count = count_chat_tokens(prompt)

    if final_token_count > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Prompt contains {final_token_count} tokens, "
            f"exceeding the {MAX_INPUT_TOKENS}-token limit. "
            "Reduce the context budget."
        )

    print("\n--- PROMPT DIAGNOSTICS ---")
    print("Original question:", question)
    print("Retrieved chunks:", len(retrieved_documents))
    print("Chunks included:", len(selected_parts))
    print("Final prompt tokens (with rules):", final_token_count)
    print("Input token limit:", MAX_INPUT_TOKENS)

    print("\n--- SELECTED EVIDENCE ---")

    for i, part in enumerate(selected_parts, start=1):
      print(f"\nEvidence {i}:")
      print(part)

    return prompt

### Step 7: Generate the answer

The `generate_answer` function formats and tokenizes the prompt, checks its length, and sends it to SEA-LION. It generates a response without random sampling and decodes only the new answer tokens.

In [29]:
# Generate an answer using the language model.

def generate_answer(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS
):
    """
    Generate an answer without silently truncating the prompt.
    """

    # Format the prompt with the chatbot rules.
    chat_text = to_chat_text(prompt)

    inputs = tokenizer(
        chat_text,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=False
    )

    input_length = inputs["input_ids"].shape[1]

    print("\n--- GENERATION DIAGNOSTICS ---")
    print("Actual input tokens:", input_length)
    print("Maximum input tokens:", MAX_INPUT_TOKENS)

    if input_length > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Input contains {input_length} tokens, "
            f"but the limit is {MAX_INPUT_TOKENS}."
        )

    # Move the inputs to the model device.
    inputs = {
        key: value.to(llm.device)
        for key, value in inputs.items()
    }

    # Generate an answer without random sampling.
    with torch.no_grad():

        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id
        )

    # Remove the input tokens from the generated output.
    new_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    if not answer:
        return (
            "The model generated an empty answer. "
            "Inspect the prompt and retrieved evidence."
        )

    return answer

### Step 8: Run the full RAG process

The `rag_query` function checks the question, retrieves relevant chunks from FAISS, builds the evidence-based prompt, and asks SEA-LION to answer. It returns the answer, retrieved sources, and prompt. It also handles empty questions and missing evidence.

In [30]:
# Combine retrieval, prompting, and answer generation.


def rag_query(question, top_k=3):
    """
    General-purpose RAG question-answering function.

    Workflow:
    1. Retrieve relevant document chunks.
    2. Build a token-limited prompt.
    3. Generate an answer using the language model.
    4. Return the answer, retrieved sources, and prompt.
    """

    # Check that the question is not empty.
    question = question.strip()

    if not question:
        return (
            "Please enter a valid question.",
            [],
            ""
        )

    # Find relevant chunks from FAISS.
    retrieved_documents = retrieve_documents(
        question,
        top_k=top_k
    )

    # Respond if no chunks were found.
    if not retrieved_documents:
        return (
            "I cannot find the answer in the provided document.",
            [],
            ""
        )

    # Build the prompt from the retrieved evidence.
    prompt = create_rag_prompt(
        question,
        retrieved_documents
    )

    # Generate an answer using SEA-LION.
    answer = generate_answer(prompt)

    # Handle an empty generated answer.
    if not answer or not answer.strip():
        answer = (
            "The model could not generate an answer. "
            "Please review the retrieved evidence."
        )

    # Return the answer, sources, and prompt.
    return (
        answer.strip(),
        retrieved_documents,
        prompt
    )

### Step 9: Ask questions in the chat loop

This cell lets us ask questions repeatedly in Colab. After each question, it prints the answer, retrieved passages, and prompt. We can type `sources` or `prompt` to review the last result, or `exit` to stop.

In [ ]:
# Run the question and answer loop.

print("\n")
print("=" * 70)
print("CONTINUOUS RAG QUESTION ANSWERING")
print("=" * 70)

print("Ask questions about the indexed document.")
print("Type 'exit' to terminate.")
print("Type 'sources' to display sources from the last query.")
print("Type 'prompt' to display the last generated prompt.")


# Save the last answer, sources, and prompt.
last_answer = None
last_sources = []
last_prompt = None


while True:

    print("\n")

    user_question = input(
        "Question: "
    ).strip()

    # Stop the loop when the user types exit.

    if user_question.lower() == "exit":

        print("\nRAG session terminated.")
        break

    # Ask the user to enter a question.

    if not user_question:

        print("Please enter a question.")
        continue

    # Show the sources from the previous question.

    if user_question.lower() == "sources":

        if not last_sources:
            print("No previous query sources are available.")
            continue

        print("\n" + "=" * 70)
        print("SOURCES FROM LAST QUERY")
        print("=" * 70)

        for result in last_sources:

            print(
                f"\nRank {result['rank']} | "
                f"Page {result['page']} | "
                f"Chunk {result['chunk_id']} | "
                f"Similarity {result['score']:.4f}"
            )

            print("-" * 70)
            print(result["text"])

        continue

    # Show the previous prompt.

    if user_question.lower() == "prompt":

        if last_prompt is None:
            print("No previous query prompt is available.")
            continue

        print("\n" + "=" * 70)
        print("LAST RAG PROMPT")
        print("=" * 70)

        print("[SYSTEM]\n" + SYSTEM_MESSAGE)
        print("\n[USER]\n" + last_prompt)

        continue

    # Process the user question.

    try:

        answer, retrieved_documents, prompt = rag_query(
            user_question,
            top_k=TOP_K
        )

        last_answer = answer
        last_sources = retrieved_documents
        last_prompt = prompt

        # Print the generated answer.

        print("\n")
        print("=" * 70)
        print("RAG ANSWER")
        print("=" * 70)

        print(answer)

        # Print the retrieved source chunks.

        print("\n")
        print("=" * 70)
        print("RETRIEVED SOURCES")
        print("=" * 70)

        for result in retrieved_documents:

            print(
                f"\n[{result['rank']}] "
                f"Page {result['page']} "
                f"| Chunk {result['chunk_id']} "
                f"| Similarity {result['score']:.4f}"
            )

            print("-" * 70)
            print(result["text"])

        # Print the prompt used for the answer.

        print("\n")
        print("=" * 70)
        print("RAG PROMPT SENT TO LLM")
        print("=" * 70)

        print("[SYSTEM]\n" + SYSTEM_MESSAGE)
        print("\n[USER]\n" + prompt)

    except Exception as e:

        print("\nERROR:")
        print(str(e))

        print("\nThe question could not be processed.")



CONTINUOUS RAG QUESTION ANSWERING
Ask questions about the indexed document.
Type 'exit' to terminate.
Type 'sources' to display sources from the last query.
Type 'prompt' to display the last generated prompt.


Question: Is Fido.ph included in the SEC's online lending records, and which company is it under?

--- PROMPT DIAGNOSTICS ---
Original question: Is Fido.ph included in the SEC's online lending records, and which company is it under?
Retrieved chunks: 3
Chunks included: 3
Final prompt tokens (with rules): 830
Input token limit: 4096

--- SELECTED EVIDENCE ---

Evidence 1:
[Page 15, Chunk 28]
LendSuri | Researcher-prepared record from the existing v5 corpus 15 LendSuri / Member 2 A Current OLP registry snapshot (2026-01-26) Record ID: CUR2026_0047 Record date: 2026-01-26 Platform or entity: www.Fido.ph Company: Fidoph Lending Corporation Evidence category: CURRENT_RECORDED_OLP_SNAPSHOT_2026 Classification: RECORDED_IN_2026_01_26_SNAPSHOT Evidence priority: CURRENT_POSITIVE_SNAPS

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.




RAG ANSWER
Fido.ph is recorded in the SEC's online lending records under the company name Fidoph Lending Corporation (SEC registration no.: CS201825101). This information is confirmed in both Page 15, Chunk 28 and Page 14, Chunk 26 of the provided evidence.


RETRIEVED SOURCES

[1] Page 15 | Chunk 28 | Similarity 0.6641
----------------------------------------------------------------------
LendSuri | Researcher-prepared record from the existing v5 corpus 15 LendSuri / Member 2 A Current OLP registry snapshot (2026-01-26) Record ID: CUR2026_0047 Record date: 2026-01-26 Platform or entity: www.Fido.ph Company: Fidoph Lending Corporation Evidence category: CURRENT_RECORDED_OLP_SNAPSHOT_2026 Classification: RECORDED_IN_2026_01_26_SNAPSHOT Evidence priority: CURRENT_POSITIVE_SNAPSHOT RAG source text (v5 normalized text) SEC OLP current-registry snapshot dated 2026-01-26. Platform: www.Fido.ph. Company: Fidoph Lending Corporation. SEC registration no.: CS201825101. Form 1 submission date(s

### Step 10: End the notebook

This cell prints a final message when the RAG program has finished.

In [ ]:
# Print the program end message.

print("\n")
print("=" * 70)
print("RAG PROGRAM FINISHED")
print("=" * 70)